In [82]:
# ---------
# CIR MODEL
# ---------

In [83]:
function PoissonRandN(lambda)
    if lambda<50
        u=rand()
        i=0
        p=exp(-lambda)
        F=p
        while u>F
            p=lambda*p/(i+1)
            F=F+p
            i=i+1
        end
        return i
    else
        k=floor(Int,lambda+sqrt(lambda)*randn()+0.5)
        return max(k,0)
    end
end

PoissonRandN (generic function with 1 method)

In [84]:
function nonCentralChiSqRand(k,lambda)
    m=PoissonRandN(lambda/2)
    x=0
    for i in 1:k+2*m
        x=x+randn()^2
    end
    return(x)
end

nonCentralChiSqRand (generic function with 1 method)

In [85]:
function CIRPath(rzero, alpha, b, sigma, T, n)
    h = T / n
    path = Array{Float64}(undef, n + 1)
    path[1] = rzero
    r = rzero

    c = sigma^2 * (1 - exp(-alpha * h)) / (4 * alpha)
    k = 4 * alpha * b / sigma^2

    for i in 1:n
        lambda = 4 * alpha * exp(-alpha * h) * r / (sigma^2 * (1 - exp(-alpha * h)))
        r = c * nonCentralChiSqRand(k, lambda)
        path[i + 1] = r
    end

    return path
end

CIRPath (generic function with 1 method)

In [86]:
# --------------
# BASE CMO MODEL
# --------------

In [87]:
function CMOprice(m, L, D, prepayRate, rzero, alpha, b, sigma, N)
    monthly_payment = L * m / (1 - (1 + m)^(-D))

    total_price = 0.0
    total_life_months = 0.0

    for i in 1:N
        balance = L

        # Simulate CIR rates and convert to monthly rates
        rates = CIRPath(rzero, alpha, b, sigma, D / 12, D)
        rates = rates / 12

        path_pv = 0.0
        discount_factor = 1.0
        months_used = 0

        for n in 1:D
            if balance <= 0
                break
            end

            # Mortgage cash flow calculations
            interest = m * balance
            principal = monthly_payment - interest
            prepayment = prepayRate * balance
            total_principal = principal + prepayment
            total_principal = min(total_principal, balance) # Prevent overpaying beyond remaining balance
            cash_flow = interest + total_principal

            # Path-wise discounting
            discount_factor = discount_factor / (1 + rates[n])
            path_pv += discount_factor * cash_flow

            # Update remaining balance
            balance -= total_principal
            months_used += 1
        end

        total_price += path_pv
        total_life_months += months_used
    end

    avg_price = total_price / N
    avg_prepay_rate = prepayRate
    avg_life_months = total_life_months / N
    avg_life_years = avg_life_months / 12

    println("CMO price estimate              = ", avg_price)
    println("Average prepayment rate/month  = ", avg_prepay_rate)
    println("Average mortgage life (months) = ", avg_life_months)
    println("Average mortgage life (years)  = ", avg_life_years)

    return (
        price = avg_price,
        avg_prepay_rate = avg_prepay_rate,
        avg_life_months = avg_life_months,
        avg_life_years = avg_life_years
    )
end

CMOprice (generic function with 1 method)

In [88]:
# ---------------------------
# STOCHASTIC PREPAYMENT MODEL
# ---------------------------

In [89]:
function CMOprice_prepay(m, L, D, prepay_a, prepay_b, rzero, alpha, b, sigma, N)
    monthly_payment = L * m / (1 - (1 + m)^(-D))

    total_price = 0.0
    total_avg_prepay_rate = 0.0
    total_life_months = 0.0

    for i in 1:N
        balance = L

        # Simulate CIR rates and convert to monthly rates
        rates = CIRPath(rzero, alpha, b, sigma, D / 12, D)
        rates = rates / 12

        path_pv = 0.0
        discount_factor = 1.0
        path_prepay_rate_sum = 0.0
        months_used = 0

        for n in 1:D
            if balance <= 0
                break
            end

            # Mortgage cash flow calculations
            interest = m * balance
            principal = monthly_payment - interest

            # Stochastic prepayment rate
            incentive = max(m - rates[n], 0.0)
            prepay_rate_n = prepay_a * exp(prepay_b * incentive)
            prepayment = prepay_rate_n * balance

            total_principal = principal + prepayment
            # Prevent overpaying beyond remaining balance
            total_principal = min(total_principal, balance)

            cash_flow = interest + total_principal

            # Path-wise discounting
            discount_factor = discount_factor / (1 + rates[n])
            path_pv += discount_factor * cash_flow

            # Update remaining balance
            balance -= total_principal
            
            path_prepay_rate_sum += prepay_rate_n
            months_used += 1
        end

        total_price += path_pv
        total_avg_prepay_rate += path_prepay_rate_sum / months_used
        total_life_months += months_used
    end

    avg_price = total_price / N
    avg_prepay_rate = total_avg_prepay_rate / N
    avg_life_months = total_life_months / N
    avg_life_years = avg_life_months / 12

    println("CMO price estimate              = ", avg_price)
    println("Average prepayment rate/month  = ", avg_prepay_rate)
    println("Average mortgage life (months) = ", avg_life_months)
    println("Average mortgage life (years)  = ", avg_life_years)

    return (
        price = avg_price,
        avg_prepay_rate = avg_prepay_rate,
        avg_life_months = avg_life_months,
        avg_life_years = avg_life_years
    )
end

CMOprice_prepay (generic function with 1 method)

In [90]:
# --------------------------------------------------
# STOCHASTIC PREPAYMENT AND STOCHASTIC DEFAULT MODEL
# --------------------------------------------------

In [91]:
function CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a, default_b, rzero, alpha, b, sigma, N)
    monthly_payment = L * m / (1 - (1 + m)^(-D))

    total_price = 0.0
    total_avg_prepay_rate = 0.0
    total_avg_default_rate = 0.0
    total_life_months = 0.0

    for i in 1:N
        balance = L

        # Simulate CIR rates and convert to monthly rates
        rates = CIRPath(rzero, alpha, b, sigma, D / 12, D)
        rates = rates / 12

        path_pv = 0.0
        discount_factor = 1.0
        path_prepay_rate_sum = 0.0
        path_default_rate_sum = 0.0
        months_used = 0

        for n in 1:D
            if balance <= 0
                break
            end

            # Mortgage cash flow calculations
            interest = m * balance
            principal = monthly_payment - interest

            # Stochastic prepayment rate
            prepay_incentive = max(m - rates[n], 0.0)
            prepay_rate_n = prepay_a * exp(prepay_b * prepay_incentive)
            prepayment = prepay_rate_n * balance

            # Stochastic default rate
            default_incentive = max(rates[n] - m, 0.0)
            default_rate_n = default_a * exp(default_b * default_incentive)
            default = default_rate_n * balance

            total_principal_reduction = principal + prepayment + default

            # Prevent overpaying/defaulting beyond remaining balance
            if total_principal_reduction > balance
                scale = balance / total_principal_reduction
                principal *= scale
                prepayment *= scale
                default *= scale
                total_principal_reduction = balance
            end

            cash_flow = interest + principal + prepayment

            # Path-wise discounting
            discount_factor = discount_factor / (1 + rates[n])
            path_pv += discount_factor * cash_flow

            # Update remaining balance
            balance -= total_principal_reduction

            path_prepay_rate_sum += prepay_rate_n
            path_default_rate_sum += default_rate_n
            months_used += 1
        end

        total_price += path_pv
        total_avg_prepay_rate += path_prepay_rate_sum / months_used
        total_avg_default_rate += path_default_rate_sum / months_used
        total_life_months += months_used
    end

    avg_price = total_price / N
    avg_prepay_rate = total_avg_prepay_rate / N
    avg_default_rate = total_avg_default_rate / N
    avg_life_months = total_life_months / N
    avg_life_years = avg_life_months / 12

    println("CMO price estimate              = ", avg_price)
    println("Average prepayment rate/month  = ", avg_prepay_rate)
    println("Average default rate/month     = ", avg_default_rate)
    println("Average mortgage life (months) = ", avg_life_months)
    println("Average mortgage life (years)  = ", avg_life_years)

    return (
        price = avg_price,
        avg_prepay_rate = avg_prepay_rate,
        avg_default_rate = avg_default_rate,
        avg_life_months = avg_life_months,
        avg_life_years = avg_life_years
    )
end

CMOprice_prepay_default (generic function with 1 method)

In [92]:
# -------
# TESTING
# -------

In [93]:
# Base model

CMOprice(0.065/12, 10_000_000, 360,     # Mortgage parameters
         0.002,                         # Constant prepayment parameter
         0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.1362810800595596e7
Average prepayment rate/month  = 0.002
Average mortgage life (months) = 229.0
Average mortgage life (years)  = 19.083333333333332


(price = 1.1362810800595596e7, avg_prepay_rate = 0.002, avg_life_months = 229.0, avg_life_years = 19.083333333333332)

In [94]:
# Stochastic prepayment model - low prepayment sensitivity

CMOprice_prepay(0.065/12, 10_000_000, 360,     # Mortgage parameters
                0.002, 40.0,                   # Prepayment parameters
                0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.1344642174457984e7
Average prepayment rate/month  = 0.002114045045262399
Average mortgage life (months) = 224.5718
Average mortgage life (years)  = 18.714316666666665


(price = 1.1344642174457984e7, avg_prepay_rate = 0.002114045045262399, avg_life_months = 224.5718, avg_life_years = 18.714316666666665)

In [95]:
# Stochastic prepayment model - moderate prepayment sensitivity

CMOprice_prepay(0.065/12, 10_000_000, 360,     # Mortgage parameters
                0.002, 80.0,                   # Prepayment parameters
                0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.1309059962226795e7
Average prepayment rate/month  = 0.0022343523705862013
Average mortgage life (months) = 220.6191
Average mortgage life (years)  = 18.384925


(price = 1.1309059962226795e7, avg_prepay_rate = 0.0022343523705862013, avg_life_months = 220.6191, avg_life_years = 18.384925)

In [96]:
# Stochastic prepayment model - high prepayment sensitivity

CMOprice_prepay(0.065/12, 10_000_000, 360,     # Mortgage parameters
                0.002, 150.0,                  # Prepayment parameters
                0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.1270683189790903e7
Average prepayment rate/month  = 0.0024707082915158677
Average mortgage life (months) = 213.4416
Average mortgage life (years)  = 17.7868


(price = 1.1270683189790903e7, avg_prepay_rate = 0.0024707082915158677, avg_life_months = 213.4416, avg_life_years = 17.7868)

In [97]:
# Stochastic prepayment model - higher baseline prepayment

CMOprice_prepay(0.065/12, 10_000_000, 360,     # Mortgage parameters
                0.004, 80.0,                   # Prepayment parameters
                0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.1027752574425858e7
Average prepayment rate/month  = 0.004477513355136907
Average mortgage life (months) = 171.6307
Average mortgage life (years)  = 14.302558333333332


(price = 1.1027752574425858e7, avg_prepay_rate = 0.004477513355136907, avg_life_months = 171.6307, avg_life_years = 14.302558333333332)

In [98]:
# Stochastic prepayment and default model - moderate prepayment sensitivity, moderate default sensitivity

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 100.0,                 # Default parameters
                       0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.0803182885488277e7
Average prepayment rate/month  = 0.002236203138383104
Average default rate/month     = 0.0005005928415078046
Average mortgage life (months) = 206.4194
Average mortgage life (years)  = 17.201616666666666


(price = 1.0803182885488277e7, avg_prepay_rate = 0.002236203138383104, avg_default_rate = 0.0005005928415078046, avg_life_months = 206.4194, avg_life_years = 17.201616666666666)

In [99]:
# Stochastic prepayment and default model - moderate prepayment sensitivity, high default sensitivity

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 150.0,                 # Default parameters
                       0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.0801142790526783e7
Average prepayment rate/month  = 0.002236353824247513
Average default rate/month     = 0.0005008974486425198
Average mortgage life (months) = 206.4286
Average mortgage life (years)  = 17.202383333333334


(price = 1.0801142790526783e7, avg_prepay_rate = 0.002236353824247513, avg_default_rate = 0.0005008974486425198, avg_life_months = 206.4286, avg_life_years = 17.202383333333334)

In [100]:
# Stochastic prepayment and default model - moderate prepayment sensitivity, higher baseline default

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.001, 100.0,                  # Default parameters
                       0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.0348475402425185e7
Average prepayment rate/month  = 0.0022353985124546737
Average default rate/month     = 0.0010011622303912132
Average mortgage life (months) = 194.5701
Average mortgage life (years)  = 16.214175


(price = 1.0348475402425185e7, avg_prepay_rate = 0.0022353985124546737, avg_default_rate = 0.0010011622303912132, avg_life_months = 194.5701, avg_life_years = 16.214175)

In [101]:
# Stochastic prepayment and default model - moderate prepayment sensitivity, very high baseline default

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.002, 100.0,                  # Default parameters
                       0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 9.59632617026287e6
Average prepayment rate/month  = 0.002238305960240505
Average default rate/month     = 0.0020021577859796434
Average mortgage life (months) = 175.5514
Average mortgage life (years)  = 14.629283333333333


(price = 9.59632617026287e6, avg_prepay_rate = 0.002238305960240505, avg_default_rate = 0.0020021577859796434, avg_life_months = 175.5514, avg_life_years = 14.629283333333333)

In [102]:
# Stochastic prepayment and default model - moderate prepayment and default sensitivities, low interest rates

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 100.0,                 # Default parameters
                       0.03, 0.3, 0.03, 0.03, 10000)  # CIR parameters / N

CMO price estimate              = 1.220391867107507e7
Average prepayment rate/month  = 0.002527266663867243
Average default rate/month     = 0.0005000000729615999
Average mortgage life (months) = 199.5086
Average mortgage life (years)  = 16.625716666666666


(price = 1.220391867107507e7, avg_prepay_rate = 0.002527266663867243, avg_default_rate = 0.0005000000729615999, avg_life_months = 199.5086, avg_life_years = 16.625716666666666)

In [103]:
# Stochastic prepayment and default model - moderate prepayment and default sensitivities, high interest rates

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 100.0,                 # Default parameters
                       0.06, 0.3, 0.07, 0.03, 10000)  # CIR parameters / N

CMO price estimate              = 9.514910409513984e6
Average prepayment rate/month  = 0.0020359179198905426
Average default rate/month     = 0.0005240106844107673
Average mortgage life (months) = 211.3364
Average mortgage life (years)  = 17.611366666666665


(price = 9.514910409513984e6, avg_prepay_rate = 0.0020359179198905426, avg_default_rate = 0.0005240106844107673, avg_life_months = 211.3364, avg_life_years = 17.611366666666665)

In [104]:
# Stochastic prepayment and default model - moderate prepayment and default sensitivities, increased rate volatility

CMOprice_prepay_default(0.065/12, 10_000_000, 360,    # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 100.0,                 # Default parameters
                       0.045, 0.3, 0.05, 0.06, 10000) # CIR parameters / N

CMO price estimate              = 1.0895685685871601e7
Average prepayment rate/month  = 0.0022821785177022614
Average default rate/month     = 0.0005069042761276768
Average mortgage life (months) = 205.3208
Average mortgage life (years)  = 17.110066666666665


(price = 1.0895685685871601e7, avg_prepay_rate = 0.0022821785177022614, avg_default_rate = 0.0005069042761276768, avg_life_months = 205.3208, avg_life_years = 17.110066666666665)

In [105]:
# Stochastic prepayment and default model - moderate prepayment and default sensitivities, low mortgage rate

CMOprice_prepay_default(0.05/12, 10_000_000, 360,     # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 100.0,                 # Default parameters
                       0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 9.7066953571959e6
Average prepayment rate/month  = 0.0020548926893444687
Average default rate/month     = 0.0005116736611487265
Average mortgage life (months) = 221.8554
Average mortgage life (years)  = 18.48795


(price = 9.7066953571959e6, avg_prepay_rate = 0.0020548926893444687, avg_default_rate = 0.0005116736611487265, avg_life_months = 221.8554, avg_life_years = 18.48795)

In [106]:
# Stochastic prepayment and default model - moderate prepayment and default sensitivities, high mortgage rate

CMOprice_prepay_default(0.08/12, 10_000_000, 360,     # Mortgage parameters
                       0.002, 80.0,                   # Prepayment parameters
                       0.0005, 100.0,                 # Default parameters
                       0.045, 0.3, 0.05, 0.03, 10000) # CIR parameters / N

CMO price estimate              = 1.1803507043521108e7
Average prepayment rate/month  = 0.002469669954303263
Average default rate/month     = 0.0005000096499163823
Average mortgage life (months) = 190.1948
Average mortgage life (years)  = 15.849566666666666


(price = 1.1803507043521108e7, avg_prepay_rate = 0.002469669954303263, avg_default_rate = 0.0005000096499163823, avg_life_months = 190.1948, avg_life_years = 15.849566666666666)

In [107]:
# -----------------------------------------------------------------
# GREEKS (using central finite differences and common random seeds)
# -----------------------------------------------------------------

In [108]:
using Random

# Epsilon sizes
eps_r0 = 0.001          # 10 basis point bump in annualized initial interest rate
eps_sigma = 0.001       # Bump in interest rate volatility
eps_m = 0.0001          # Bump in monthly mortgage rate
eps_prepay_a = 0.0001   # Bump in base prepayment rate
eps_default_a = 0.00005 # Bump in base default rate

# Benchmark parameters
m = 0.065 / 12
L = 10_000_000
D = 360
prepay_a = 0.002
prepay_b = 80.0
default_a = 0.0005
default_b = 100.0
rzero = 0.045
alpha = 0.3
b = 0.05
sigma = 0.03
N = 10000

10000

In [109]:
# Greek with respect to r0

Random.seed!(1234)
P_r0_plus = CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a, default_b,
    rzero + eps_r0, alpha, b, sigma, N).price

Random.seed!(1234)
P_r0_minus = CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a, default_b,
    rzero - eps_r0, alpha, b, sigma, N).price

dP_dr0 = (P_r0_plus - P_r0_minus) / (2 * eps_r0)

println("\nGreek estimate:")
println("dP/dr0 = ", dP_dr0)
println("10 bp effect ≈ ", dP_dr0 * 0.001)

CMO price estimate              = 1.0780531489580538e7
Average prepayment rate/month  = 0.0022336749007736815
Average default rate/month     = 0.000500613795860657
Average mortgage life (months) = 206.5279
Average mortgage life (years)  = 17.21065833333333
CMO price estimate              = 1.0826919328894686e7
Average prepayment rate/month  = 0.00223879493484239
Average default rate/month     = 0.0005005938898500536
Average mortgage life (months) = 206.3046
Average mortgage life (years)  = 17.19205

Greek estimate:
dP/dr0 = -2.3193919657073915e7
10 bp effect ≈ -23193.919657073915


In [110]:
# Greek with respect to sigma

Random.seed!(1234)
P_sigma_plus = CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a, default_b,
    rzero, alpha, b, sigma + eps_sigma, N).price

Random.seed!(1234)
P_sigma_minus = CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a, default_b,
    rzero, alpha, b, sigma - eps_sigma, N).price

dP_dsigma = (P_sigma_plus - P_sigma_minus) / (2 * eps_sigma)

println("\nGreek estimate:")
println("dP/dsigma = ", dP_dsigma)
println("per 1% increase in volatility ≈ ", dP_dsigma * 0.01)

CMO price estimate              = 1.0791869888123332e7
Average prepayment rate/month  = 0.002234319721557986
Average default rate/month     = 0.0005007559214546312
Average mortgage life (months) = 206.4697
Average mortgage life (years)  = 17.205808333333334
CMO price estimate              = 1.0788133846219374e7
Average prepayment rate/month  = 0.0022325984672249195
Average default rate/month     = 0.00050053594717833
Average mortgage life (months) = 206.5068
Average mortgage life (years)  = 17.2089

Greek estimate:
dP/dsigma = 1.8680209519788623e6
per 1% increase in volatility ≈ 18680.209519788623


In [111]:
# Greek with respect to mortgage rate m

Random.seed!(1234)
P_m_plus = CMOprice_prepay_default(m + eps_m, L, D, prepay_a, prepay_b, default_a, default_b,
    rzero, alpha, b, sigma, N).price

Random.seed!(1234)
P_m_minus = CMOprice_prepay_default(m - eps_m, L, D, prepay_a, prepay_b, default_a, default_b,
    rzero, alpha, b, sigma, N).price

dP_dm = (P_m_plus - P_m_minus) / (2 * eps_m)

println("\nGreek estimate:")
println("dP/dm = ", dP_dm)
println("10 bp effect ≈ ", dP_dm * 0.001)

CMO price estimate              = 1.0890196282091457e7
Average prepayment rate/month  = 0.0022539760608813026
Average default rate/month     = 0.0005004584621602574
Average mortgage life (months) = 205.094
Average mortgage life (years)  = 17.091166666666666
CMO price estimate              = 1.0721700616074083e7
Average prepayment rate/month  = 0.002219116334531897
Average default rate/month     = 0.000500812596112242
Average mortgage life (months) = 207.7096
Average mortgage life (years)  = 17.30913333333333

Greek estimate:
dP/dm = 8.424783300868701e8
10 bp effect ≈ 842478.3300868701


In [112]:
# Greek with respect to prepay_a

Random.seed!(1234)
P_prepay_a_plus = CMOprice_prepay_default(m, L, D, prepay_a + eps_prepay_a, prepay_b, default_a, default_b,
    rzero, alpha, b, sigma, N).price

Random.seed!(1234)
P_prepay_a_minus = CMOprice_prepay_default(m, L, D, prepay_a - eps_prepay_a, prepay_b, default_a, default_b,
    rzero, alpha, b, sigma, N).price

dP_dprepay_a = (P_prepay_a_plus - P_prepay_a_minus) / (2 * eps_prepay_a)

println("\nGreek estimate:")
println("dP/d(prepay_a) = ", dP_dprepay_a)

CMO price estimate              = 1.0795810811097883e7
Average prepayment rate/month  = 0.002348407990568178
Average default rate/month     = 0.0005006089294494835
Average mortgage life (months) = 203.5705
Average mortgage life (years)  = 16.964208333333335
CMO price estimate              = 1.0817041085459618e7
Average prepayment rate/month  = 0.002124412188270552
Average default rate/month     = 0.0005006165994748589
Average mortgage life (months) = 209.3591
Average mortgage life (years)  = 17.446591666666666

Greek estimate:
dP/d(prepay_a) = -1.0615137180867605e8


In [113]:
# Greek with respect to default_a

Random.seed!(1234)
P_default_a_plus = CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a + eps_default_a, default_b,
    rzero, alpha, b, sigma, N).price

Random.seed!(1234)
P_default_a_minus = CMOprice_prepay_default(m, L, D, prepay_a, prepay_b, default_a - eps_default_a, default_b,
    rzero, alpha, b, sigma, N).price

dP_ddefault_a = (P_default_a_plus - P_default_a_minus) / (2 * eps_default_a)

println("\nGreek estimate:")
println("dP/d(default_a) = ", dP_ddefault_a)

CMO price estimate              = 1.0758707691761497e7
Average prepayment rate/month  = 0.002236475812537362
Average default rate/month     = 0.0005506721242939327
Average mortgage life (months) = 205.1228
Average mortgage life (years)  = 17.093566666666668
CMO price estimate              = 1.0854407642063044e7
Average prepayment rate/month  = 0.0022363360704020807
Average default rate/month     = 0.00045055283141038813
Average mortgage life (months) = 207.7097
Average mortgage life (years)  = 17.309141666666665

Greek estimate:
dP/d(default_a) = -9.56999503015466e8
